# 04 - Scoring New Cases and Presentation Dashboard Data
## Surprise Medical Bill Risk Predictor - SDSU Capstone

This notebook:
1. Loads trained models from `artifacts/`
2. Scores all records with predicted risk and exposure
3. Creates dashboard CSV files for the Streamlit app
4. Demonstrates sample predictions for presentation
5. Outputs `artifacts/state_dashboard.csv` and `artifacts/provider_dashboard.csv`

In [ ]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Load models
print('Loading trained models...')
clf = joblib.load('artifacts/risk_classifier.joblib')
reg = joblib.load('artifacts/exposure_regressor.joblib')

with open('artifacts/model_metrics.json') as f:
    metrics = json.load(f)

print('Models loaded successfully')
print(f"Classification AUC: {metrics['classification_auc']:.4f}")
print(f"Regression MAE:     ${metrics['regression_mae']:,.2f}")
print(f"Regression RMSE:    ${metrics['regression_rmse']:,.2f}")
print(f"Regression R²:      {metrics['regression_r2']:.4f}")

In [ ]:
# Load feature data
print('Loading feature data...')
df = pd.read_parquet('data/model_features.parquet')
print(f'Loaded: {len(df):,} records')

numeric_features = [
    'average_covered_charges', 'average_total_payments', 'Billed amount', 'Medicare payment',
    'charge_ratio_inpatient', 'payment_gap_inpatient', 'charge_ratio_outpatient', 'payment_gap_outpatient',
    'blended_charge_ratio', 'blended_payment_gap', 'log_avg_covered', 'log_avg_payments',
    'log_billed_amount', 'log_medicare_payment', 'log_blended_gap',
    'is_er', 'is_imaging', 'is_outpatient', 'is_surgery', 'high_complexity_drg',
    'provider_avg_gap', 'provider_gap_std', 'provider_avg_ratio', 'provider_record_count',
    'state_avg_gap', 'state_avg_ratio', 'state_median_payment', 'state_record_count',
    'provider_risk_index', 'state_risk_index', 'service_risk_weight',
]
categorical_features = ['provider_state', 'service_type']
text_feature = 'drg_text'

X = df[numeric_features + categorical_features + [text_feature]].copy()

## 1. Score All Records

In [ ]:
print('Scoring all records (this may take 1-2 minutes)...')

# Score in batches to avoid memory issues
BATCH_SIZE = 10000
all_risk_probs = []
all_exposures = []

for start in range(0, len(X), BATCH_SIZE):
    batch = X.iloc[start:start+BATCH_SIZE]
    risk_probs = clf.predict_proba(batch)[:, 1]
    exposures = reg.predict(batch)
    all_risk_probs.extend(risk_probs)
    all_exposures.extend(exposures)
    if (start // BATCH_SIZE) % 5 == 0:
        print(f'  Scored {min(start+BATCH_SIZE, len(X)):,}/{len(X):,} records...')

df['predicted_risk_prob'] = all_risk_probs
df['predicted_exposure']  = np.clip(all_exposures, 0, None)
df['predicted_risk_label']= (df['predicted_risk_prob'] >= 0.5).astype(int)
df['risk_band'] = pd.cut(
    df['predicted_risk_prob'],
    bins=[0, 0.33, 0.67, 1.0],
    labels=['Low', 'Moderate', 'High']
)

print(f'\nScoring complete!')
print(f'Risk band distribution:')
print(df['risk_band'].value_counts())

## 2. Create State Dashboard CSV

In [ ]:
print('Creating state dashboard data...')

state_dashboard = df.groupby('provider_state').agg(
    total_records        = ('provider_id', 'count'),
    unique_providers     = ('provider_id', 'nunique'),
    avg_risk_prob        = ('predicted_risk_prob', 'mean'),
    high_risk_count      = ('predicted_risk_label', 'sum'),
    avg_exposure         = ('predicted_exposure', 'mean'),
    median_exposure      = ('predicted_exposure', 'median'),
    avg_charge_ratio     = ('blended_charge_ratio', 'mean'),
    avg_payment_gap      = ('blended_payment_gap', 'mean'),
    avg_covered_charges  = ('average_covered_charges', 'mean'),
    avg_total_payments   = ('average_total_payments', 'mean'),
).reset_index()

state_dashboard.columns = [
    'state', 'total_records', 'unique_providers', 'avg_risk_pct', 'high_risk_count',
    'avg_exposure', 'median_exposure', 'avg_charge_ratio', 'avg_payment_gap',
    'avg_covered_charges', 'avg_total_payments'
]
state_dashboard = state_dashboard.sort_values('avg_risk_pct', ascending=False)

state_dashboard.to_csv('artifacts/state_dashboard.csv', index=False)
print(f'State dashboard saved: {len(state_dashboard)} states')
print(state_dashboard.head(10).to_string(index=False))

## 3. Create Provider Dashboard CSV

In [ ]:
print('Creating provider dashboard data...')

provider_dashboard = df.groupby(['provider_id', 'provider_name', 'provider_state']).agg(
    total_records     = ('provider_id', 'count'),
    avg_risk_prob     = ('predicted_risk_prob', 'mean'),
    high_risk_count   = ('predicted_risk_label', 'sum'),
    avg_exposure      = ('predicted_exposure', 'mean'),
    avg_charge_ratio  = ('blended_charge_ratio', 'mean'),
    avg_payment_gap   = ('blended_payment_gap', 'mean'),
    provider_risk_index = ('provider_risk_index', 'mean'),
).reset_index()

provider_dashboard = provider_dashboard[provider_dashboard['total_records'] >= 5]
provider_dashboard = provider_dashboard.sort_values('avg_risk_prob', ascending=False)

provider_dashboard.to_csv('artifacts/provider_dashboard.csv', index=False)
print(f'Provider dashboard saved: {len(provider_dashboard)} providers')
print('\nTop 10 highest risk providers:')
print(provider_dashboard.head(10)[['provider_name','provider_state','avg_risk_prob','avg_exposure','avg_charge_ratio']].to_string(index=False))

## 4. Sample Predictions (For Presentation Demo)

In [ ]:
print('=== SAMPLE PREDICTIONS FOR PRESENTATION ===')
print()

eps = 1e-6

def score_scenario(name, avg_covered, avg_payments, billed, medicare, state, service, is_er=0, is_surgery=0, is_imaging=0):
    charge_ratio_inp = avg_covered / (avg_payments + eps)
    payment_gap_inp  = avg_covered - avg_payments
    charge_ratio_out = billed / (medicare + eps)
    payment_gap_out  = billed - medicare

    # Get state defaults from data
    state_data = df[df['provider_state'] == state]
    if len(state_data) > 0:
        state_defaults = state_data[[
            'provider_avg_gap','provider_gap_std','provider_avg_ratio','provider_record_count',
            'state_avg_gap','state_avg_ratio','state_median_payment','state_record_count',
            'provider_risk_index','state_risk_index'
        ]].median().to_dict()
    else:
        state_defaults = {'provider_avg_gap':5000,'provider_gap_std':2000,'provider_avg_ratio':1.5,
                          'provider_record_count':50,'state_avg_gap':4000,'state_avg_ratio':1.4,
                          'state_median_payment':8000,'state_record_count':1000,
                          'provider_risk_index':0.4,'state_risk_index':0.5}

    blended_ratio = 0.6*charge_ratio_inp + 0.4*charge_ratio_out
    blended_gap   = 0.6*payment_gap_inp  + 0.4*payment_gap_out

    scenario = {
        'average_covered_charges': avg_covered,
        'average_total_payments':  avg_payments,
        'Billed amount':           billed,
        'Medicare payment':        medicare,
        'charge_ratio_inpatient':  charge_ratio_inp,
        'payment_gap_inpatient':   payment_gap_inp,
        'charge_ratio_outpatient': charge_ratio_out,
        'payment_gap_outpatient':  payment_gap_out,
        'blended_charge_ratio':    blended_ratio,
        'blended_payment_gap':     blended_gap,
        'log_avg_covered':         np.log1p(avg_covered),
        'log_avg_payments':        np.log1p(avg_payments),
        'log_billed_amount':       np.log1p(billed),
        'log_medicare_payment':    np.log1p(medicare),
        'log_blended_gap':         np.log1p(max(blended_gap, 0)),
        'is_er':                   is_er,
        'is_imaging':              is_imaging,
        'is_outpatient':           1 - is_er,
        'is_surgery':              is_surgery,
        'high_complexity_drg':     is_surgery,
        'service_risk_weight':     0.35*is_er + 0.25*is_surgery + 0.15*is_imaging + 0.15*is_surgery,
        'provider_state':          state,
        'service_type':            service,
        'drg_text':                service.lower().replace('-', ' '),
        **state_defaults
    }

    X_s = pd.DataFrame([scenario])[numeric_features + categorical_features + [text_feature]]
    risk_prob = float(clf.predict_proba(X_s)[:, 1][0])
    exposure  = max(0, float(reg.predict(X_s)[0]))

    risk_band = 'HIGH' if risk_prob >= 0.67 else ('MODERATE' if risk_prob >= 0.34 else 'LOW')
    print(f'Scenario: {name}')
    print(f'  State: {state} | Service: {service}')
    print(f'  Risk Probability: {risk_prob:.1%} → {risk_band} RISK')
    print(f'  Estimated Exposure: ${exposure:,.2f}')
    print(f'  Charge Ratio: {blended_ratio:.2f}x | Payment Gap: ${blended_gap:,.0f}')
    print()

# Scenario 1: ER visit in Texas (high risk state)
score_scenario('Emergency Room - Texas',
    avg_covered=62000, avg_payments=18000, billed=8500, medicare=2200,
    state='TX', service='emergency department', is_er=1)

# Scenario 2: Planned hip replacement (moderate/high risk)
score_scenario('Hip Replacement Surgery - New York',
    avg_covered=85000, avg_payments=32000, billed=4500, medicare=1800,
    state='NY', service='inpatient surgery', is_surgery=1)

# Scenario 3: Routine outpatient - California (lower risk)
score_scenario('Routine Outpatient - California',
    avg_covered=18000, avg_payments=12000, billed=1200, medicare=800,
    state='CA', service='same-day procedure')

## 5. Presentation Summary Visualizations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# 1. Top risky states
top_states = state_dashboard.head(15)
axes[0].barh(range(len(top_states)), top_states['avg_risk_pct'], color='tomato', alpha=0.8)
axes[0].set_yticks(range(len(top_states)))
axes[0].set_yticklabels(top_states['state'])
axes[0].set_xlabel('Avg Surprise Bill Risk')
axes[0].set_title('Top 15 States by Risk')

# 2. Risk band distribution
risk_counts = df['risk_band'].value_counts().sort_index()
colors = {'Low': 'steelblue', 'Moderate': 'orange', 'High': 'tomato'}
bar_colors = [colors.get(str(b), 'gray') for b in risk_counts.index]
axes[1].bar(risk_counts.index.astype(str), risk_counts.values, color=bar_colors, alpha=0.8, edgecolor='black')
axes[1].set_title('Risk Band Distribution (All Records)')
axes[1].set_ylabel('Number of Cases')
for i, v in enumerate(risk_counts.values):
    axes[1].text(i, v + 200, f'{v:,}', ha='center', fontsize=9)

# 3. Exposure by service type
service_exposure = df.groupby('service_type')['predicted_exposure'].mean().sort_values(ascending=False)
axes[2].barh(range(len(service_exposure)), service_exposure.values, color='mediumorchid', alpha=0.8)
axes[2].set_yticks(range(len(service_exposure)))
axes[2].set_yticklabels(service_exposure.index)
axes[2].set_xlabel('Avg Predicted Exposure ($)')
axes[2].set_title('Average Exposure by Service Type')

plt.tight_layout()
plt.savefig('artifacts/04_dashboard_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('Dashboard summary plot saved to artifacts/04_dashboard_summary.png')

In [ ]:
print('='*60)
print('NOTEBOOK 4 COMPLETE - DASHBOARD DATA READY')
print('='*60)
print(f'\nFiles created:')
print(f'  artifacts/state_dashboard.csv     ({len(state_dashboard)} states)')
print(f'  artifacts/provider_dashboard.csv  ({len(provider_dashboard)} providers)')
print(f'  artifacts/04_dashboard_summary.png')
print(f'\nDataset summary:')
print(f'  Total records scored:  {len(df):,}')
print(f'  High risk cases:       {(df["risk_band"]=="High").sum():,} ({(df["risk_band"]=="High").mean()*100:.1f}%)')
print(f'  Moderate risk cases:   {(df["risk_band"]=="Moderate").sum():,} ({(df["risk_band"]=="Moderate").mean()*100:.1f}%)')
print(f'  Low risk cases:        {(df["risk_band"]=="Low").sum():,} ({(df["risk_band"]=="Low").mean()*100:.1f}%)')
print(f'\n✅ Ready to run the Streamlit dashboard!')
print(f'   Run: streamlit run streamlit_app/app.py')